# Shannon-Prime Model-Pack Calibration Sweep

Automated PPL benchmarking across multiple models and compression paths.

**What this does:**
1. Builds `sp-engine` with CUDA (auto-detected GPU arch)
2. Scans Google Drive for all `.gguf` files and presents a checklist
3. Runs a full calibration sweep per model: baseline, ship, ship+cauchy, sqfree, sqfree+spinor, hierarchical
4. Auto-adjusts ctx/chunks and n-gpu-layers based on GPU VRAM
5. Outputs structured JSON results + summary table with PASS/FAIL verdicts
6. Optionally runs FP8 kernel test on Ampere+ GPUs

**Budget thresholds (drift/baseline):**

| Path | Max drift |
|---|---|
| Ship (default) | 5% |
| Sqfree | 10% |
| Sqfree+spinor | 15% |
| Hierarchical | 15% |

**Requirements:** Colab Pro GPU runtime (T4 / L4 / A100). Google Drive with GGUF models.

## 1 -- Mount Drive & Create Folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, shutil, time, json, re, glob
from datetime import datetime, timezone

ROOT = '/content/drive/MyDrive/Model Tests'
DIRS = {
    'engine': f'{ROOT}/engine',
    'models': f'{ROOT}/models',
    'corpus': f'{ROOT}/corpus',
    'logs':   f'{ROOT}/logs',
    'results': f'{ROOT}/calibration_results',
    'build':  '/content/build',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

for k, v in DIRS.items():
    os.environ[k.upper()] = v
os.environ['ENGINE']  = DIRS['engine']
os.environ['MODELS']  = DIRS['models']
os.environ['CORPUS']  = DIRS['corpus']
os.environ['LOGS']    = DIRS['logs']
os.environ['BUILD']   = DIRS['build']
os.environ['RESULTS'] = DIRS['results']

# Shannon-Prime engine env vars
os.environ['SP_ENGINE_BACKEND'] = 'gpu'
os.environ['SHANNON_PRIME_GPU_CACHE'] = '1'

print('Folders ready:')
for k, v in DIRS.items():
    print(f'  {k:10s} -> {v}')

## 2 -- GPU Detection & VRAM Budget

In [ ]:
def detect_gpu():
    """Detect GPU name, VRAM, compute capability, and SM architecture."""
    info = {}
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
             '--format=csv,noheader,nounits'],
            text=True
        ).strip().split(', ')
        info['name'] = out[0].strip()
        info['vram_mb'] = int(float(out[1].strip()))
        cap = out[2].strip()
        info['compute_cap'] = cap
        info['sm'] = int(cap.replace('.', ''))
    except Exception as e:
        print(f'GPU detection failed: {e}')
        info = {'name': 'Unknown', 'vram_mb': 16000, 'compute_cap': '7.5', 'sm': 75}

    # Friendly name for results
    name_lower = info['name'].lower()
    if 'a100' in name_lower:
        info['gpu_tag'] = 'A100'
    elif 'l4' in name_lower:
        info['gpu_tag'] = 'L4'
    elif 't4' in name_lower:
        info['gpu_tag'] = 'T4'
    elif 'h100' in name_lower:
        info['gpu_tag'] = 'H100'
    elif 'v100' in name_lower:
        info['gpu_tag'] = 'V100'
    else:
        info['gpu_tag'] = info['name']

    info['is_ampere_plus'] = info['sm'] >= 80
    return info

GPU = detect_gpu()
print(f"GPU: {GPU['name']}")
print(f"VRAM: {GPU['vram_mb']} MB")
print(f"Compute: sm_{GPU['sm']}")
print(f"Ampere+: {GPU['is_ampere_plus']}")
print(f"Tag: {GPU['gpu_tag']}")

## 3 -- Install Dependencies & Verify Toolchain

In [ ]:
%%bash
set -e
echo '--- CUDA ---'
nvcc --version | tail -1
echo '--- CMake ---'
cmake --version | head -1
echo '--- GCC ---'
g++ --version | head -1
echo '--- GPU ---'
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 4 -- Clone / Update shannon-prime-engine

In [ ]:
%%bash
set -e
cd "$ENGINE"

if [ ! -d .git ]; then
    echo '>>> Fresh clone'
    cd ..
    rm -rf engine
    git clone --recurse-submodules https://github.com/nihilistau/shannon-prime-engine.git engine
else
    echo '>>> Updating existing clone'
    git fetch origin main
    git reset --hard origin/main
    git submodule update --init --recursive
fi

echo '--- Latest commits ---'
git log --oneline -5
echo '--- Submodule status ---'
git submodule status

## 5 -- Configure & Build (CUDA)

In [ ]:
%%bash
set -e

# Auto-detect GPU compute capability
GPU_ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.')
echo "Detected GPU arch: sm_${GPU_ARCH}"

cmake -B "$BUILD" -S "$ENGINE" \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES="${GPU_ARCH}" \
    -DSP_ENGINE_WITH_CUDA=ON \
    -DSP_ENGINE_BUILD_CLI=ON \
    -DSP_ENGINE_BUILD_TESTS=ON \
    2>&1 | tail -20

echo ''
echo '--- Build ---'
cmake --build "$BUILD" --config Release -j$(nproc) 2>&1 | tail -30

echo ''
echo '--- Binaries ---'
ls -lh "$BUILD"/bin/sp-engine 2>/dev/null || echo 'sp-engine not found!'
echo ''
echo '--- Smoke test ---'
"$BUILD"/bin/sp-engine --version 2>&1 || "$BUILD"/bin/sp-engine help 2>&1 | head -5

## 6 -- Run Test Suite

In [ ]:
%%bash
set -e
cd "$BUILD"
ctest -C Release --output-on-failure 2>&1 | tail -40

## 7 -- Download wikitext-2-raw Corpus

In [ ]:
import os

WIKI = f"{os.environ['CORPUS']}/wiki.test.raw"

if os.path.exists(WIKI) and os.path.getsize(WIKI) > 100000:
    print(f'Corpus already present: {os.path.getsize(WIKI):,} bytes')
else:
    print('Downloading wikitext-2-raw...')
    subprocess.run(['pip', 'install', '-q', 'datasets'], check=True)
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    with open(WIKI, 'w') as f:
        f.write('\n'.join(ds['text']))
    print(f'Saved: {os.path.getsize(WIKI):,} bytes ({len(ds)} entries)')

os.environ['WIKI'] = WIKI

## 8 -- Scan Drive for GGUF Models & Select

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display

def scan_gguf_files(search_roots=None):
    """Recursively scan for .gguf files under given roots."""
    if search_roots is None:
        search_roots = [
            os.environ['MODELS'],
            '/content/drive/MyDrive/Model Tests',
            '/content/drive/MyDrive/models',
        ]
    found = {}
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            for fn in filenames:
                if fn.endswith('.gguf'):
                    full = os.path.join(dirpath, fn)
                    if fn not in found:  # deduplicate by filename
                        found[fn] = full
    return found

GGUF_MAP = scan_gguf_files()

if not GGUF_MAP:
    print('No .gguf files found on Drive!')
    print('Upload models to Google Drive under "Model Tests/models/" and re-run.')
else:
    print(f'Found {len(GGUF_MAP)} GGUF file(s):\n')
    checkboxes = {}
    for fn in sorted(GGUF_MAP.keys()):
        path = GGUF_MAP[fn]
        size_gb = os.path.getsize(path) / (1024**3)
        cb = widgets.Checkbox(value=True, description=f'{fn} ({size_gb:.1f} GB)',
                              layout=widgets.Layout(width='auto'),
                              style={'description_width': 'initial'})
        checkboxes[fn] = cb
        display(cb)

    print('\nUncheck models you want to skip, then run the next cell.')

In [ ]:
# Collect selected models
SELECTED_MODELS = []

try:
    # If widgets were used
    for fn, cb in checkboxes.items():
        if cb.value:
            SELECTED_MODELS.append({'filename': fn, 'path': GGUF_MAP[fn]})
except NameError:
    # Fallback: select all found models
    for fn, path in GGUF_MAP.items():
        SELECTED_MODELS.append({'filename': fn, 'path': path})

print(f'Selected {len(SELECTED_MODELS)} model(s) for calibration:')
for m in SELECTED_MODELS:
    print(f'  {m["filename"]}')

## 9 -- Model Metadata Inference

In [ ]:
def infer_model_metadata(filename):
    """Infer arch, quant, and size class from GGUF filename."""
    fn = filename.lower()
    meta = {'arch': 'unknown', 'quant': 'unknown', 'size_class': 'small'}

    # Architecture detection
    if 'llama' in fn:
        meta['arch'] = 'llama'
    elif 'qwen3.6' in fn or 'qwen3-next' in fn or 'qwen35moe' in fn:
        meta['arch'] = 'qwen35moe'
    elif 'qwen3' in fn or 'qwen' in fn:
        meta['arch'] = 'qwen3'
    elif 'gemma-4' in fn or 'gemma4' in fn:
        meta['arch'] = 'gemma4'
    elif 'gemma-3' in fn or 'gemma3' in fn:
        meta['arch'] = 'gemma3'
    elif 'gemma' in fn:
        meta['arch'] = 'gemma3'  # default gemma -> gemma3
    elif 'phi' in fn:
        meta['arch'] = 'phi3'
    elif 'mistral' in fn:
        meta['arch'] = 'mistral3'

    # Quant detection
    quant_patterns = [
        'IQ4_NL', 'IQ4_XS', 'IQ3_XXS', 'IQ3_S', 'IQ2_XXS',
        'Q8_0', 'Q6_K', 'Q5_K_S', 'Q5_K_M', 'Q5_K_P',
        'Q4_K_S', 'Q4_K_M', 'Q4_K_L', 'Q4_0', 'Q3_K_S', 'Q3_K_M', 'Q3_K_L',
        'Q2_K', 'F16', 'F32',
    ]
    for qp in quant_patterns:
        if qp.lower() in fn:
            meta['quant'] = qp
            break

    # Size class: determines ctx/chunks budget
    # Extract parameter count from filename
    size_match = re.search(r'(\d+)[bB]', fn)
    if size_match:
        param_b = int(size_match.group(1))
        if param_b <= 10:
            meta['size_class'] = 'small'    # 8B and under
        elif param_b <= 15:
            meta['size_class'] = 'medium'   # 13B-ish
        else:
            meta['size_class'] = 'large'    # 31B, 35B, etc.
    elif '35b' in fn or '31b' in fn or '27b' in fn or '70b' in fn:
        meta['size_class'] = 'large'

    return meta


def compute_run_params(meta, gpu_info):
    """Compute ctx, chunks, and n-gpu-layers based on model size and GPU VRAM."""
    vram = gpu_info['vram_mb']
    params = {}

    if meta['size_class'] == 'small':
        params['ctx'] = 2048
        params['chunks'] = 8
        params['n_gpu_layers'] = -1  # full offload
    elif meta['size_class'] == 'medium':
        params['ctx'] = 1536
        params['chunks'] = 6
        params['n_gpu_layers'] = -1 if vram >= 24000 else 32
    else:  # large
        params['ctx'] = 1024
        params['chunks'] = 4
        if vram >= 80000:      # A100 80GB
            params['n_gpu_layers'] = -1
        elif vram >= 40000:    # A100 40GB
            params['n_gpu_layers'] = 48
        elif vram >= 24000:    # L4 24GB
            params['n_gpu_layers'] = 28
        elif vram >= 16000:    # T4 16GB
            params['n_gpu_layers'] = 16
        else:
            params['n_gpu_layers'] = 8

    return params


# Annotate selected models
for m in SELECTED_MODELS:
    m['meta'] = infer_model_metadata(m['filename'])
    m['params'] = compute_run_params(m['meta'], GPU)
    print(f"{m['filename']}:")
    print(f"  arch={m['meta']['arch']}  quant={m['meta']['quant']}  size={m['meta']['size_class']}")
    print(f"  ctx={m['params']['ctx']}  chunks={m['params']['chunks']}  n_gpu_layers={m['params']['n_gpu_layers']}")
    print()

## 10 -- Calibration Sweep Engine

Core runner: executes `sp-engine perplexity` for each model x path combination and captures structured results.

In [ ]:
import subprocess, time, json, re, os
from datetime import datetime, timezone

SP_ENGINE = f"{os.environ['BUILD']}/bin/sp-engine"
WIKI = os.environ['WIKI']

# Calibration paths to test
CALIBRATION_PATHS = [
    {
        'name': 'baseline',
        'flags': ['--model-preset', 'off'],
        'threshold': None,  # reference
    },
    {
        'name': 'ship',
        'flags': ['--cache', '--model-preset', 'auto'],
        'threshold': 0.05,
    },
    {
        'name': 'ship_cauchy',
        'flags': ['--cache', '--cauchy-mode', '2'],
        'threshold': 0.05,
    },
    {
        'name': 'sqfree',
        'flags': ['--cache', '--sqfree'],
        'threshold': 0.10,
    },
    {
        'name': 'sqfree_spinor',
        'flags': ['--cache', '--sqfree', '--spinor'],
        'threshold': 0.15,
    },
    {
        'name': 'hier',
        'flags': ['--cache', '--hierarchical'],
        'threshold': 0.15,
    },
]


def run_perplexity(model_path, ctx, chunks, extra_flags, n_gpu_layers=-1, timeout_sec=1800):
    """Run sp-engine perplexity and return (ppl, wall_seconds, raw_output)."""
    cmd = [
        SP_ENGINE, 'perplexity',
        '--model', model_path,
        '--ctx', str(ctx),
        '--chunks', str(chunks),
    ]

    if n_gpu_layers != -1:
        cmd.extend(['--n-gpu-layers', str(n_gpu_layers)])

    cmd.extend(extra_flags)
    cmd.append(WIKI)

    env = os.environ.copy()
    env['SP_ENGINE_BACKEND'] = 'gpu'
    env['SHANNON_PRIME_GPU_CACHE'] = '1'

    print(f'    CMD: {" ".join(cmd)}')
    t0 = time.time()

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True,
            timeout=timeout_sec, env=env
        )
        wall = time.time() - t0
        output = result.stdout + '\n' + result.stderr

        # Parse PPL from output -- look for last PPL_running or final_perplexity
        ppl = None
        # Try final_perplexity first
        m = re.search(r'final_perplexity[=:]\s*(\d+\.\d+)', output)
        if m:
            ppl = float(m.group(1))
        else:
            # Fall back to last PPL_running
            matches = re.findall(r'PPL_running=(\d+\.\d+)', output)
            if matches:
                ppl = float(matches[-1])
            else:
                # Try other patterns
                matches = re.findall(r'perplexity\s*[=:]\s*(\d+\.\d+)', output, re.IGNORECASE)
                if matches:
                    ppl = float(matches[-1])

        if result.returncode != 0 and ppl is None:
            print(f'    ERROR (exit {result.returncode}):')
            print(output[-500:] if len(output) > 500 else output)

        return ppl, wall, output

    except subprocess.TimeoutExpired:
        wall = time.time() - t0
        print(f'    TIMEOUT after {wall:.0f}s')
        return None, wall, 'TIMEOUT'
    except Exception as e:
        wall = time.time() - t0
        print(f'    EXCEPTION: {e}')
        return None, wall, str(e)


print('Calibration engine ready.')
print(f'sp-engine: {SP_ENGINE}')
print(f'Corpus: {WIKI}')
print(f'Paths to test: {[p["name"] for p in CALIBRATION_PATHS]}')

## 11 -- Run Full Calibration Sweep

Iterates over all selected models and all calibration paths. Saves results incrementally to Drive.

In [ ]:
ALL_RESULTS = []
RESULTS_FILE = f"{os.environ['RESULTS']}/calibration_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

total_runs = len(SELECTED_MODELS) * len(CALIBRATION_PATHS)
run_idx = 0

for model in SELECTED_MODELS:
    fn = model['filename']
    meta = model['meta']
    params = model['params']

    print(f'\n{"="*70}')
    print(f'MODEL: {fn}')
    print(f'  arch={meta["arch"]}  quant={meta["quant"]}  ctx={params["ctx"]}  chunks={params["chunks"]}  ngl={params["n_gpu_layers"]}')
    print(f'{"="*70}')

    baseline_ppl = None

    for path_cfg in CALIBRATION_PATHS:
        run_idx += 1
        path_name = path_cfg['name']

        # Skip hierarchical for models with ctx < 512
        if path_name == 'hier' and params['ctx'] < 512:
            print(f'\n  [{run_idx}/{total_runs}] SKIP {path_name} (ctx {params["ctx"]} < 512)')
            continue

        print(f'\n  [{run_idx}/{total_runs}] {path_name}...')

        ppl, wall, raw = run_perplexity(
            model_path=model['path'],
            ctx=params['ctx'],
            chunks=params['chunks'],
            extra_flags=path_cfg['flags'],
            n_gpu_layers=params['n_gpu_layers'],
        )

        if path_name == 'baseline' and ppl is not None:
            baseline_ppl = ppl

        # Compute drift
        drift_pct = None
        verdict = None
        if ppl is not None and baseline_ppl is not None and path_name != 'baseline':
            drift_pct = ((ppl / baseline_ppl) - 1.0) * 100.0
            threshold = path_cfg['threshold']
            if threshold is not None:
                verdict = 'PASS' if (drift_pct / 100.0) <= threshold else 'FAIL'

        result = {
            'model': fn,
            'arch': meta['arch'],
            'quant': meta['quant'],
            'gpu': GPU['gpu_tag'],
            'path': path_name,
            'ctx': params['ctx'],
            'chunks': params['chunks'],
            'n_gpu_layers': params['n_gpu_layers'],
            'ppl': ppl,
            'drift_pct': round(drift_pct, 4) if drift_pct is not None else None,
            'threshold_pct': (path_cfg['threshold'] * 100) if path_cfg['threshold'] else None,
            'verdict': verdict,
            'wall_seconds': round(wall, 1),
            'timestamp': datetime.now(timezone.utc).isoformat(),
        }

        ALL_RESULTS.append(result)

        # Print inline summary
        ppl_str = f'{ppl:.4f}' if ppl is not None else 'ERROR'
        drift_str = f'+{drift_pct:.2f}%' if drift_pct is not None else ''
        verdict_str = f'[{verdict}]' if verdict else ''
        print(f'    PPL={ppl_str}  {drift_str}  {verdict_str}  ({wall:.0f}s)')

        # Save log
        log_fn = f"{os.environ['LOGS']}/ppl_{meta['arch']}_{meta['quant']}_{path_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        with open(log_fn, 'w') as f:
            f.write(raw)

        # Incremental JSON save
        with open(RESULTS_FILE, 'w') as f:
            json.dump(ALL_RESULTS, f, indent=2)

print(f'\n\nCalibration sweep complete: {len(ALL_RESULTS)} results')
print(f'Saved to: {RESULTS_FILE}')

## 12 -- Summary Table

In [ ]:
from collections import defaultdict

# Group by model
by_model = defaultdict(list)
for r in ALL_RESULTS:
    by_model[r['model']].append(r)

print(f'\n{"="*100}')
print(f' SHANNON-PRIME CALIBRATION RESULTS -- {GPU["gpu_tag"]} -- {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'{"="*100}\n')

for model_name, results in by_model.items():
    baseline = next((r for r in results if r['path'] == 'baseline'), None)
    base_ppl = baseline['ppl'] if baseline and baseline['ppl'] else None

    print(f'Model: {model_name}')
    arch_info = results[0]
    print(f'  arch={arch_info["arch"]}  quant={arch_info["quant"]}  ctx={arch_info["ctx"]}  chunks={arch_info["chunks"]}  ngl={arch_info["n_gpu_layers"]}')
    print()
    print(f'  {"Path":<20s} {"PPL":>10s} {"Drift":>10s} {"Budget":>10s} {"Verdict":>10s} {"Time":>8s}')
    print(f'  {"-"*18:<20s} {"-"*10:>10s} {"-"*10:>10s} {"-"*10:>10s} {"-"*10:>10s} {"-"*8:>8s}')

    for r in results:
        ppl_s = f'{r["ppl"]:.4f}' if r['ppl'] is not None else 'ERROR'
        drift_s = f'+{r["drift_pct"]:.2f}%' if r['drift_pct'] is not None else ('ref' if r['path'] == 'baseline' else 'N/A')
        budget_s = f'<={r["threshold_pct"]:.0f}%' if r['threshold_pct'] is not None else '---'
        verdict_s = r['verdict'] if r['verdict'] else '---'
        time_s = f'{r["wall_seconds"]:.0f}s'
        print(f'  {r["path"]:<20s} {ppl_s:>10s} {drift_s:>10s} {budget_s:>10s} {verdict_s:>10s} {time_s:>8s}')

    # Overall verdict for this model
    verdicts = [r['verdict'] for r in results if r['verdict'] is not None]
    if verdicts:
        if all(v == 'PASS' for v in verdicts):
            overall = 'ALL PASS'
        else:
            fails = sum(1 for v in verdicts if v == 'FAIL')
            overall = f'{fails} FAIL'
        print(f'\n  >>> Overall: {overall}')
    print(f'\n{"-"*100}\n')

## 13 -- Per-Model Calibration Verdicts

In [ ]:
CALIBRATION_LEDGER = []

for model_name, results in by_model.items():
    baseline = next((r for r in results if r['path'] == 'baseline'), None)
    base_ppl = baseline['ppl'] if baseline else None
    arch = results[0]['arch']
    quant = results[0]['quant']

    ship_r = next((r for r in results if r['path'] == 'ship'), None)
    sqfree_r = next((r for r in results if r['path'] == 'sqfree'), None)
    spinor_r = next((r for r in results if r['path'] == 'sqfree_spinor'), None)
    hier_r = next((r for r in results if r['path'] == 'hier'), None)

    entry = {
        'model': model_name,
        'arch': arch,
        'quant': quant,
        'baseline_ppl': base_ppl,
        'ship_ppl': ship_r['ppl'] if ship_r else None,
        'ship_drift': ship_r['drift_pct'] if ship_r else None,
        'ship_verdict': ship_r['verdict'] if ship_r else None,
        'sqfree_drift': sqfree_r['drift_pct'] if sqfree_r else None,
        'sqfree_verdict': sqfree_r['verdict'] if sqfree_r else None,
        'spinor_drift': spinor_r['drift_pct'] if spinor_r else None,
        'spinor_verdict': spinor_r['verdict'] if spinor_r else None,
        'hier_drift': hier_r['drift_pct'] if hier_r else None,
        'hier_verdict': hier_r['verdict'] if hier_r else None,
        'gpu': GPU['gpu_tag'],
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }

    # Overall: CALIBRATED if ship passes, PROVISIONAL if ship untested, FAIL if ship fails
    if entry['ship_verdict'] == 'PASS':
        entry['status'] = 'CALIBRATED'
    elif entry['ship_verdict'] == 'FAIL':
        entry['status'] = 'FAIL'
    else:
        entry['status'] = 'PROVISIONAL'

    CALIBRATION_LEDGER.append(entry)

    status_icon = {'CALIBRATED': '[OK]', 'FAIL': '[!!]', 'PROVISIONAL': '[??]'}
    print(f"{status_icon.get(entry['status'], '   ')} {model_name}: {entry['status']}")
    if entry['ship_drift'] is not None:
        print(f"     Ship drift: +{entry['ship_drift']:.2f}% (budget: <=5%)")

print(f'\nLedger: {len(CALIBRATION_LEDGER)} models')

## 14 -- FP8 Integration Test (Ampere+ only)

If the GPU is sm_80+, test FP8 encode/decode kernel.

In [ ]:
if GPU['is_ampere_plus']:
    print(f"GPU is Ampere+ (sm_{GPU['sm']}), running FP8 integration test...\n")

    # Try the dedicated FP8 test binary first
    fp8_test = f"{os.environ['BUILD']}/bin/test_fp8_kernel"
    sp_engine = f"{os.environ['BUILD']}/bin/sp-engine"

    fp8_result = None

    if os.path.exists(fp8_test):
        print(f'Running: {fp8_test}')
        res = subprocess.run([fp8_test], capture_output=True, text=True, timeout=120)
        print(res.stdout)
        if res.stderr:
            print(res.stderr)
        fp8_result = 'PASS' if res.returncode == 0 else 'FAIL'
    else:
        # Try sp-engine fp8-test subcommand
        print(f'No dedicated test binary. Trying: sp-engine fp8-test')
        res = subprocess.run(
            [sp_engine, 'fp8-test'],
            capture_output=True, text=True, timeout=120
        )
        output = res.stdout + '\n' + res.stderr
        print(output)

        if res.returncode == 0:
            fp8_result = 'PASS'
        elif 'unknown command' in output.lower() or 'not found' in output.lower():
            print('FP8 test subcommand not available in this build.')
            fp8_result = 'SKIPPED'
        else:
            fp8_result = 'FAIL'

    # Also test via ctest if available
    print('\n--- CTest FP8 filter ---')
    res2 = subprocess.run(
        ['ctest', '-C', 'Release', '-R', 'fp8', '--output-on-failure'],
        capture_output=True, text=True, timeout=120,
        cwd=os.environ['BUILD']
    )
    print(res2.stdout)
    if res2.stderr:
        print(res2.stderr)

    print(f'\nFP8 integration test: {fp8_result}')
else:
    print(f"GPU is sm_{GPU['sm']} (pre-Ampere), skipping FP8 test.")
    fp8_result = 'SKIPPED (pre-Ampere)'

## 15 -- Save Results to Drive

In [ ]:
import json, os
from datetime import datetime, timezone

results_dir = os.environ['RESULTS']
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

# --- JSON output ---
json_path = f'{results_dir}/calibration_{ts}.json'
full_output = {
    'meta': {
        'gpu': GPU,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'sp_engine': SP_ENGINE,
        'corpus': WIKI,
        'fp8_test': fp8_result if 'fp8_result' in dir() else 'NOT_RUN',
    },
    'results': ALL_RESULTS,
    'ledger': CALIBRATION_LEDGER,
}
with open(json_path, 'w') as f:
    json.dump(full_output, f, indent=2)
print(f'JSON saved: {json_path}')

# --- Markdown summary ---
md_path = f'{results_dir}/calibration_{ts}.md'
lines = [
    f'# Shannon-Prime Calibration Report',
    f'',
    f'**Date:** {datetime.now().strftime("%Y-%m-%d %H:%M UTC")}',
    f'**GPU:** {GPU["name"]} ({GPU["vram_mb"]} MB, sm_{GPU["sm"]})',
    f'**Models tested:** {len(SELECTED_MODELS)}',
    f'**Total runs:** {len(ALL_RESULTS)}',
    f'',
]

# FP8 status
if 'fp8_result' in dir():
    lines.append(f'**FP8 integration test:** {fp8_result}')
    lines.append('')

# Verdicts table
lines.append('## Calibration Verdicts')
lines.append('')
lines.append('| Model | Arch | Quant | Status | Ship drift | Sqfree drift | Spinor drift | Hier drift |')
lines.append('|---|---|---|---|---|---|---|---|')
for e in CALIBRATION_LEDGER:
    def fmt_drift(d):
        return f'+{d:.2f}%' if d is not None else 'N/A'
    lines.append(
        f'| {e["model"]} | {e["arch"]} | {e["quant"]} | **{e["status"]}** '
        f'| {fmt_drift(e["ship_drift"])} | {fmt_drift(e["sqfree_drift"])} '
        f'| {fmt_drift(e["spinor_drift"])} | {fmt_drift(e["hier_drift"])} |'
    )
lines.append('')

# Full results table
lines.append('## Detailed Results')
lines.append('')
lines.append('| Model | Path | PPL | Drift | Budget | Verdict | Time |')
lines.append('|---|---|---|---|---|---|---|')
for r in ALL_RESULTS:
    ppl_s = f'{r["ppl"]:.4f}' if r['ppl'] is not None else 'ERROR'
    drift_s = f'+{r["drift_pct"]:.2f}%' if r['drift_pct'] is not None else ('ref' if r['path'] == 'baseline' else 'N/A')
    budget_s = f'<={r["threshold_pct"]:.0f}%' if r['threshold_pct'] is not None else '---'
    verdict_s = r['verdict'] if r['verdict'] else '---'
    lines.append(
        f'| {r["model"]} | {r["path"]} | {ppl_s} | {drift_s} | {budget_s} | {verdict_s} | {r["wall_seconds"]}s |'
    )
lines.append('')

# Budget thresholds reference
lines.append('## Budget Thresholds')
lines.append('')
lines.append('| Path | Max drift |')
lines.append('|---|---|')
lines.append('| Ship | <=5% |')
lines.append('| Sqfree | <=10% |')
lines.append('| Sqfree+spinor | <=15% |')
lines.append('| Hierarchical | <=15% |')
lines.append('')

md_content = '\n'.join(lines)
with open(md_path, 'w') as f:
    f.write(md_content)
print(f'Markdown saved: {md_path}')

# Print summary
print(f'\n--- Quick Summary ---')
for e in CALIBRATION_LEDGER:
    print(f"  {e['model']}: {e['status']}")

## 16 -- Load Previous Results (optional)

If you want to reload results from a prior run without re-running the sweep.

In [ ]:
# Set this to a previous results JSON to reload
RELOAD_PATH = None  # e.g., '/content/drive/MyDrive/Model Tests/calibration_results/calibration_20260426_120000.json'

if RELOAD_PATH and os.path.exists(RELOAD_PATH):
    with open(RELOAD_PATH) as f:
        loaded = json.load(f)
    ALL_RESULTS = loaded['results']
    CALIBRATION_LEDGER = loaded['ledger']
    print(f'Loaded {len(ALL_RESULTS)} results from {RELOAD_PATH}')
    print(f'Ledger: {len(CALIBRATION_LEDGER)} models')
    for e in CALIBRATION_LEDGER:
        print(f"  {e['model']}: {e['status']}")
elif RELOAD_PATH:
    print(f'File not found: {RELOAD_PATH}')
else:
    # List available result files
    result_files = sorted(glob.glob(f"{os.environ['RESULTS']}/calibration_*.json"))
    if result_files:
        print('Available result files:')
        for rf in result_files:
            size_kb = os.path.getsize(rf) / 1024
            print(f'  {rf}  ({size_kb:.1f} KB)')
    else:
        print('No previous results found. Run the calibration sweep first.')

## 17 -- Visualization (optional)

Bar chart of PPL drift across models and paths.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    # Filter to non-baseline results with valid PPL
    plot_data = [r for r in ALL_RESULTS if r['path'] != 'baseline' and r['drift_pct'] is not None]

    if not plot_data:
        print('No drift data to plot.')
    else:
        # Group by model
        models = list(dict.fromkeys(r['model'] for r in plot_data))
        paths = ['ship', 'ship_cauchy', 'sqfree', 'sqfree_spinor', 'hier']
        path_colors = {
            'ship': '#2196F3',
            'ship_cauchy': '#03A9F4',
            'sqfree': '#FF9800',
            'sqfree_spinor': '#F44336',
            'hier': '#9C27B0',
        }
        thresholds = {'ship': 5, 'ship_cauchy': 5, 'sqfree': 10, 'sqfree_spinor': 15, 'hier': 15}

        fig, ax = plt.subplots(figsize=(max(12, len(models) * 3), 6))

        x = np.arange(len(models))
        width = 0.15
        offsets = np.arange(len(paths)) - len(paths) / 2 + 0.5

        for i, path in enumerate(paths):
            vals = []
            colors = []
            for model in models:
                r = next((r for r in plot_data if r['model'] == model and r['path'] == path), None)
                if r:
                    vals.append(r['drift_pct'])
                    colors.append('#4CAF50' if r['verdict'] == 'PASS' else '#F44336')
                else:
                    vals.append(0)
                    colors.append('#BDBDBD')
            bars = ax.bar(x + offsets[i] * width, vals, width, label=path, color=colors, edgecolor='white')

        # Threshold lines
        ax.axhline(y=5, color='#2196F3', linestyle='--', alpha=0.5, label='Ship budget (5%)')
        ax.axhline(y=10, color='#FF9800', linestyle='--', alpha=0.5, label='Sqfree budget (10%)')
        ax.axhline(y=15, color='#F44336', linestyle='--', alpha=0.5, label='Spinor/Hier budget (15%)')

        ax.set_xlabel('Model')
        ax.set_ylabel('PPL Drift (%)')
        ax.set_title('Shannon-Prime Calibration: PPL Drift by Model and Path')
        ax.set_xticks(x)
        # Truncate long model names
        ax.set_xticklabels([m[:30] for m in models], rotation=45, ha='right')
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        chart_path = f"{os.environ['RESULTS']}/calibration_{ts}_chart.png"
        plt.savefig(chart_path, dpi=150)
        plt.show()
        print(f'Chart saved: {chart_path}')

except ImportError:
    print('matplotlib not available. Install with: pip install matplotlib')

---

## Notes

- **Budget thresholds** are from `docs/MODEL-PACK-CALIBRATION.md` in shannon-prime
- **Ship path** = VHT2 + Mobius default compression (the production path)
- **Sqfree** = square-free lattice compression (more aggressive)
- **Spinor** = spinor-augmented sqfree (most aggressive non-hierarchical)
- **Hierarchical** = multi-level cache with eviction (needs ctx>=512)
- **Cauchy mode 2** = Cauchy reset with frequency-domain detection
- **n-gpu-layers** is auto-adjusted for large models on smaller GPUs
- All results are saved incrementally so you can resume after a disconnect
- FP8 test only runs on Ampere+ (sm_80+) GPUs